In [ ]:
import torch
import torch.nn as nn
import onnx

from espertaTorch import MultiEsperta

In [ ]:
# Example configuration matching original models
ESPERTA_CONFIGS = [
    # s1 models
    ([-6.07, -1.75, 1.14, 0.56], 0.28),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
    
    # s2 models
    ([-6.07, -1.75, 1.14, 0.56], 0.35),  # w20_120
    ([-7.44, -2.99, 1.21, 0.69], 0.28),  # e40_19
    ([-5.02, -1.74, 0.64, 0.40], 0.23),  # e120_41
]

# Initialize compound model
multi_esperta = MultiEsperta(ESPERTA_CONFIGS)

# Export to ONNX
dummy_input = torch.rand((1, 4), dtype=torch.float32)
torch.onnx.export(
    multi_esperta,
    (dummy_input,),
    f"ESPERTA.onnx",  # filename of the ONNX model
    input_names=["input"],  # Rename inputs for the ONNX model
    output_names=["output"],  # Rename outputs for the ONNX model
    dynamo=False,  # True or False to select the exporter to use
)

# Compare PyTorch model to original model

In [ ]:
from esperta import esperta_original

esperta_s1_w20_120 = esperta_original([-6.07,-1.75, 1.14, 0.56], 0.28) 
esperta_s2_w20_120 = esperta_original([-6.07,-1.75, 1.14, 0.56], 0.35)

esperta_s1_e40_19 = esperta_original([-7.44,-2.99, 1.21, 0.69], 0.28) 
esperta_s2_e40_19 = esperta_original([-7.44,-2.99, 1.21, 0.69], 0.28)

esperta_s1_e120_41 = esperta_original([-5.02,-1.74, 0.64, 0.40], 0.23) 
esperta_s2_e120_41 = esperta_original([-5.02,-1.74, 0.64, 0.40], 0.23)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate random test inputs
num_samples = 100
test_inputs = torch.rand((num_samples, 4), dtype=torch.float32)

# Get outputs from PyTorch model
pytorch_outputs = multi_esperta(test_inputs).detach().numpy()

# List of original models in the same order as in ESPERTA_CONFIGS
original_models = [
    esperta_s1_w20_120,
    esperta_s1_e40_19,
    esperta_s1_e120_41,
    esperta_s2_w20_120,
    esperta_s2_e40_19,
    esperta_s2_e120_41
]

model_names = [
    "S1 W20_120",
    "S1 E40_19",
    "S1 E120_41",
    "S2 W20_120",
    "S2 E40_19",
    "S2 E120_41"
]

# Calculate outputs from original models
original_outputs = np.zeros((num_samples, len(original_models)))
for i in range(num_samples):
    input_np = test_inputs[i].numpy()
    for j, model in enumerate(original_models):
        original_outputs[i, j] = model(input_np)

# Compare outputs using MSE and check if they're the same
mse = np.mean((pytorch_outputs - original_outputs)**2, axis=0)
is_same = np.allclose(pytorch_outputs, original_outputs, rtol=1e-5, atol=1e-5)

print("Comparison of PyTorch vs Original models:")
print("=" * 50)
print(f"Are all outputs the same? {'Yes' if is_same else 'No'}")
print("-" * 50)
for i, name in enumerate(model_names):
    print(f"{name}:")
    print(f"  Mean Square Error: {mse[i]:.8e}")
    is_model_same = np.allclose(pytorch_outputs[:, i], original_outputs[:, i], rtol=1e-5, atol=1e-5)
    print(f"  Outputs match: {'Yes' if is_model_same else 'No'}")
